# NYC Mobility - Source Ingestion

## What ingestion means

Ingestion is simply how we bring source data into our storage before transforming it. We preserve the source first so we can always trace what arrived.


## Green Taxi

The monthly Parquet files were placed in the Green Taxi source folder in R2. No automated download code was included in the original notebooks, so we do not invent one here.

Files already referenced by the project:

- `green_tripdata_2026-03.parquet`
- `green_tripdata_2026-04.parquet`
- `green_tripdata_2026-05.parquet`


## Taxi Zones

The exact file `taxi_zone_lookup.csv` was manually placed in:

`/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/`

The original work does not include automated acquisition code, so we document the real landing step and do not invent a download process.


## Weather API ingestion

We use the Open-Meteo Archive API because the assignment period is historical: March through May 2026. Select one month with the `weather_month` widget so March, April, and May can be ingested independently. The response is saved as raw JSON without flattening because cleaning and reshaping belong after ingestion.


In [0]:
%python
import calendar
import hashlib
import requests
from datetime import datetime
from pathlib import Path

allowed_months = ["2026-03", "2026-04", "2026-05"]
try:
    weather_month = dbutils.widgets.get("weather_month").strip()
except Exception:
    dbutils.widgets.dropdown("weather_month", "2026-03", allowed_months, "Weather month")
    weather_month = dbutils.widgets.get("weather_month").strip()

if weather_month not in allowed_months:
    raise ValueError(f"weather_month must be one of {allowed_months}")

month_start = datetime.strptime(weather_month, "%Y-%m").date()
month_end_day = calendar.monthrange(month_start.year, month_start.month)[1]
start_date = month_start.isoformat()
end_date = month_start.replace(day=month_end_day).isoformat()

api_url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude=40.7128"
    f"&longitude=-74.006"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    f"&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m"
    f"&timezone=America%2FNew_York"
)

weather_dir = Path(
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather"
)
weather_dir.mkdir(parents=True, exist_ok=True)
output_path = weather_dir / f"open_meteo_{start_date}_{end_date}.json"

response = requests.get(api_url, timeout=60)
response.raise_for_status()
raw_content = response.content
new_hash = hashlib.sha256(raw_content).hexdigest()

if output_path.exists():
    existing_hash = hashlib.sha256(output_path.read_bytes()).hexdigest()
    if existing_hash == new_hash:
        print(f"Idempotent skip: {output_path.name} already contains the same response.")
    else:
        raise RuntimeError(
            f"{output_path.name} already exists with different content; review before replacing raw data."
        )
else:
    output_path.write_bytes(raw_content)
    print("Saved to:", output_path)

print("HTTP status:", response.status_code)
print("Selected month:", weather_month)
print("Date range:", start_date, "to", end_date)
print("Bytes:", len(raw_content))
print("SHA-256:", new_hash)

The earlier fixed-range run returned **HTTP 200** and saved `open_meteo_2026-03-01_2026-05-31.json`. The revised cell has not yet been executed in Databricks. It will save one deterministic date-range filename per selected month and skip writing when a rerun returns identical content. A same-name file with different content raises an error for manual review instead of silently overwriting raw data.


## Traffic Advisory web scraping

This is the optional bonus source. The scraper hashes the downloaded HTML before writing. If the same content already exists, it performs an idempotent skip. Only new content receives a new UTC timestamp batch ID, raw HTML file, and metadata JSON sidecar.

The verified saved batch is `20260914T040846Z`. We use these exact files for the Bronze load:

- `nyc_dot_weekend_traffic_20260914T040846Z.html`
- `nyc_dot_weekend_traffic_20260914T040846Z.metadata.json`

Scraper idempotency and Bronze-load idempotency are separate checks. The scraper skips an unchanged response by content hash. For the Bronze idempotency test, rerun the Bronze load against the same saved HTML and metadata files.


In [0]:
import hashlib
import json
import requests
from datetime import datetime, timezone
from pathlib import Path

source_url = "https://www.nyc.gov/html/dot/html/motorist/wkndtraf.shtml"

base_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory"
)

advisory_dir = Path(base_path)
advisory_dir.mkdir(parents=True, exist_ok=True)

response = requests.get(
    source_url,
    timeout=60,
    headers={
        "User-Agent": "FTW-B12-Data-Engineering-Course-Project"
    }
)

response.raise_for_status()

raw_content = response.content
content_hash = hashlib.sha256(raw_content).hexdigest()
matching_metadata = None

for metadata_path in advisory_dir.glob("*.metadata.json"):
    try:
        existing_metadata = json.loads(metadata_path.read_text())
    except (OSError, json.JSONDecodeError) as exc:
        print(f"Warning: could not inspect {metadata_path.name}: {exc}")
        continue

    existing_hash = existing_metadata.get("sha256")
    if existing_hash is None:
        existing_html = metadata_path.with_name(
            metadata_path.name.replace(".metadata.json", ".html")
        )
        if existing_html.exists():
            existing_hash = hashlib.sha256(existing_html.read_bytes()).hexdigest()

    if existing_hash == content_hash:
        matching_metadata = metadata_path
        break

if matching_metadata is not None:
    print(f"Idempotent skip: content matches {matching_metadata.name}.")
else:
    scraped_at = datetime.now(timezone.utc)
    batch_id = scraped_at.strftime("%Y%m%dT%H%M%SZ")
    html_path = advisory_dir / f"nyc_dot_weekend_traffic_{batch_id}.html"
    metadata_path = advisory_dir / f"nyc_dot_weekend_traffic_{batch_id}.metadata.json"

    html_path.write_bytes(raw_content)

    metadata = {
        "source_system": "nyc_dot",
        "source_url": source_url,
        "batch_id": batch_id,
        "ingested_at_utc": scraped_at.isoformat(),
        "http_status": response.status_code,
        "content_type": response.headers.get("Content-Type"),
        "bytes_received": len(raw_content),
        "sha256": content_hash
    }
    metadata_path.write_text(json.dumps(metadata, indent=2))

    print("HTML saved to:", html_path)
    print("Metadata saved to:", metadata_path)

print("HTTP status:", response.status_code)
print("Bytes received:", len(raw_content))
print("SHA-256:", content_hash)

The original run returned **HTTP 200**, saved 53,391 bytes of raw HTML, and wrote a 295-byte metadata file. The revised hash-based skip has not yet been executed in Databricks. BeautifulSoup inspection belongs in the source-inspection notebook. Event and road parsing does not belong in ingestion or Bronze.


## Ingestion Summary

Green Taxi Parquet files and the exact Taxi Zone CSV were manually placed in their R2 source folders. Weather ingestion now accepts one assignment month at a time and uses a deterministic date-range filename. Traffic Advisory ingestion skips content already saved under the same SHA-256 hash and timestamps only new content. Raw source responses remain uncleaned and unexpanded in ingestion. The revised Weather and Traffic cells still require Databricks execution before reporting new results.
